In [ ]:
import os
REPO_NAME = "vision_blindspots"
REPO_URL = "https://github.com/aprzezdziecka/vision_blindspots.git"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

print("Instalacja zależności...")
!pip install uv
!uv export --format requirements-txt > requirements.txt
!uv pip install --system -r requirements.txt

print("\nŚrodowisko gotowe.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/vision_blindspots_results"
if not os.path.exists(save_path):
    os.makedirs(save_path)

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.5 MB/s eta 0:00:00


In [ ]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 14.7 MB/s eta 0:00:00


In [ ]:
from transformers import CLIPProcessor, CLIPModel, AutoProcessor, AutoModel
from PIL import Image
import requests
from datasets import load_dataset
from huggingface_hub import notebook_login
import torch
import faiss
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
import pandas as pd

In [ ]:
notebook_login()

In [ ]:
def add_index(sample, idx):
    sample['id'] = idx
    return sample

In [ ]:
print("Connecting to Caltech101...")
dataset_caltech = load_dataset("flwrlabs/caltech101", split="train", streaming=True, trust_remote_code=True)

dataset_caltech = dataset_caltech.map(add_index, with_indices=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'flwrlabs/caltech101' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'flwrlabs/caltech101' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Connecting to Caltech101...


README.md: 0.00B [00:00, ?B/s]

In [ ]:
print("Connecting to ImageNet1k...")
dataset_imagenet = load_dataset("imagenet-1k", split="train", streaming=True, trust_remote_code=True)
dataset_imagenet = dataset_imagenet.map(add_index, with_indices=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imagenet-1k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imagenet-1k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Connecting to ImageNet1k...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
# print("Connecting to cc3m-wds...")
# dataset_cc3m = load_dataset("pixparse/cc3m-wds", split="train", streaming=True, trust_remote_code=True)
# print(type(dataset_cc3m))

# iterator = iter(dataset_cc3m)
# first_sample = next(iterator)

# print("\nDataset Labels:", first_sample['label'])
# first_sample['image']

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch16"
clip_model = CLIPModel.from_pretrained(model_id)
clip_processor = CLIPProcessor.from_pretrained(model_id)
clip_model.to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [ ]:
model_id = "google/siglip-base-patch16-224"
siglip_model = AutoModel.from_pretrained(model_id)
siglip_processor = AutoProcessor.from_pretrained(model_id)
siglip_model.to(device)

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features

In [ ]:
model_id = "facebook/dinov2-base"
dino_model = AutoModel.from_pretrained(model_id)
dino_processor = AutoProcessor.from_pretrained(model_id)
dino_model.to(device)

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06,

In [ ]:
def build_embeddings(dataset_iterator, model, processor, device, total_samples, batch_size=64):
    model.eval()

    # Inicjalizacja bazy FAISS
    index = None

    all_labels = []
    all_embeddings = []
    batch_images = []
    batch_labels = []

    # Batchami pobiera zdjęcia
    for i, sample in tqdm(enumerate(dataset_iterator), total=total_samples):
        image = sample['image']
        if image.mode != "RGB":
            image = image.convert("RGB")

        batch_images.append(image)
        batch_labels.append(sample['label'])
        # Pobieramy do batcha zdjecia az do 64 lub ostatni batch np zostalo 50 zdjec
        if len(batch_images) == batch_size or (i == total_samples - 1):

            inputs = processor(images=batch_images, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model(**inputs) if not hasattr(model, 'get_image_features') else model.get_image_features(**inputs)
                # clip i siglip wykorzystujemy to, bo tylko zdjecie chcemy get_image_features(**inputs)
                # dino ma to model(**inputs)

                if hasattr(outputs, 'image_embeds') and outputs.image_embeds is not None: # clip i siglip
                    features = outputs.image_embeds
                elif hasattr(outputs, 'last_hidden_state'): #dino
                    features = outputs.last_hidden_state[:, 0, :]
                else:
                    features = outputs

            normalized_features = F.normalize(features, p=2, dim=1) # normalizacja
            embeddings = normalized_features.cpu().numpy().astype('float32')

            # Jeśli to pierwsza paczka i nie mamy jeszcze bazy, sprawdzamy rozmiar i ją tworzymy
            if index is None:
                wymiar = embeddings.shape[1] # Sprawdzamy, ile liczb ma wektor, bo kazdy model zwraca rozna dlugosc wektora, sprawdzic jakie
                print(f"\nTworzę bazę FAISS")
                index = faiss.IndexFlatIP(wymiar) # IP - inner product

            index.add(embeddings)

            all_labels.extend(batch_labels)
            all_embeddings.extend(embeddings)

            batch_images = []
            batch_labels = []

        if i >= total_samples - 1:
            break

    print(f"\nLiczba wszystkich przetworzonych zdjęć: {index.ntotal}")

    return np.array(all_embeddings), np.array(all_labels), index

In [ ]:
iterator_caltech = iter(dataset_caltech)
num = dataset_caltech.info.splits['train'].num_examples

siglip_embeddings, siglip_labels, siglip_faiss_index = build_embeddings(
    dataset_iterator=iterator_caltech,
    model=siglip_model,
    processor=siglip_processor,
    device=device,
    total_samples=num,
    batch_size=64
)

  0%|          | 0/8677 [00:00<?, ?it/s]


Tworzę bazę FAISS
/nLiczba wszystkich przetworzonych zdjęć: 8677


In [ ]:
iterator_caltech = iter(dataset_caltech)

clip_embeddings, clip_labels, clip_faiss_index = build_embeddings(
    dataset_iterator=iterator_caltech,
    model=clip_model,
    processor=clip_processor,
    device=device,
    total_samples=num,
    batch_size=64
)

  0%|          | 0/8677 [00:00<?, ?it/s]


Tworzę bazę FAISS
/nLiczba wszystkich przetworzonych zdjęć: 8677


In [ ]:
iterator_caltech = iter(dataset_caltech)

dino_embeddings, dino_labels, dino_faiss_index = build_embeddings(
    dataset_iterator=iterator_caltech,
    model=dino_model,
    processor=dino_processor,
    device=device,
    total_samples=num,
    batch_size=64
)

  0%|          | 0/8677 [00:00<?, ?it/s]


Tworzę bazę FAISS
/nLiczba wszystkich przetworzonych zdjęć: 8677


In [ ]:
def search_similar(faiss_index, query_embeddings, k=5):
  D, I = faiss_index.search(query_embeddings, k)
  # dla kazdego embeddingu znajdz mi k-1najblizszych
  # D (Distances) - wyniki podobieństwa (im bliżej 1, tym bardziej podobne, bo użyłeś IndexFlatIP + normalizacji)
  # I (Indices) - identyfikatory (Twoje przypisane ID) najbliższych zdjęć

  embeddings_num = I.shape[0]

  unique_pairs = set() # Zbiór do śledzenia tego, co już widzieliśmy
  filtered_results = [] # Ostateczna lista wyników bez duplikatów

  for q_id in range(embeddings_num):
      for j in range(k):
          match_id = I[q_id, j]
          score = D[q_id, j]

          # 1. FAISS czasem zwraca -1, jeśli brakuje sąsiadów w małej bazie
          if match_id == -1:
              continue

          # 2. Usuwamy dopasowania zdjęcia do samego siebie (np. 1 do 1)
          if q_id == match_id:
              continue

          # 3. UJEDNOLICENIE PARY - klucz do usunięcia duplikatów
          # Używamy min/max lub sorted(), żeby mniejsze ID zawsze było pierwsze
          pair = (min(q_id, match_id), max(q_id, match_id))

          # 4. Sprawdzamy, czy ta para już wystąpiła
          if pair not in unique_pairs:
              unique_pairs.add(pair)
              # Dodajemy do ostatecznych wyników (Zdjecie_A, Zdjecie_B, Podobieństwo)
              filtered_results.append({
                  "id_1": pair[0],
                  "id_2": pair[1],
                  "score": score
              })
  return filtered_results

In [ ]:
clip_k_best_similar = search_similar(clip_faiss_index, clip_embeddings)

In [ ]:
def find_blind_spots(base_k_best_similar, embeddings_ref_1, embeddings_ref_2, name_base, name_ref_1, name_ref_2, threshold_similar=0.95, threshold_diff=0.6):

  base_similar = [score for score in base_k_best_similar if score['score'] > 0.95]
  result = []
  for i in range(len(base_similar)):
    current = base_similar[i]
    if current['score'] > threshold_similar:

      id_1 = current['id_1']
      id_2 = current['id_2']

      # ref1
      curr_emb_1_ref_1 = embeddings_ref_1[id_1]
      curr_emb_2_ref_1 = embeddings_ref_1[id_2]
      similarity_ref_1 = np.dot(curr_emb_1_ref_1, curr_emb_2_ref_1)
      # ref2
      curr_emb_1_ref_2 = embeddings_ref_2[id_1]
      curr_emb_2_ref_2 = embeddings_ref_2[id_2]
      similarity_ref_2 = np.dot(curr_emb_1_ref_2, curr_emb_2_ref_2)

      if similarity_ref_1 < threshold_diff or similarity_ref_2 < threshold_diff:
        result.append({'id_1': id_1, 'id_2': id_2, 'base_name': name_base, 'score_base' : current['score'], 'ref_1_name' : name_ref_1, 'score_ref_1' : similarity_ref_1, 'ref_2_name' : name_ref_2, 'score_ref_2' : similarity_ref_2})
  return pd.DataFrame(result)


In [ ]:
clip_caltech_blind_spots = find_blind_spots(clip_k_best_similar, siglip_embeddings, dino_embeddings, 'CLIP', 'SigLIP', 'DINO')
clip_caltech_blind_spots.to_csv("clip_caltech_blind_spots", sep=';', encoding='utf-8')

In [ ]:
clip_caltech_blind_spots

,id_1,id_2,base_name,score_base,ref_1_name,score_ref_1,ref_2_name,score_ref_2
0,20,465,CLIP,0.961227,SigLIP,0.425819,DINO,0.990587
1,55,7297,CLIP,0.987392,SigLIP,0.586700,DINO,0.990597
2,84,8550,CLIP,0.965167,SigLIP,0.366609,DINO,0.748084
3,84,4583,CLIP,0.963342,SigLIP,0.417315,DINO,0.788334
4,116,5039,CLIP,0.957302,SigLIP,0.139836,DINO,0.982948
...,...,...,...,...,...,...,...,...
189,3609,8550,CLIP,0.967688,SigLIP,0.494593,DINO,0.891577
190,1732,8550,CLIP,0.966574,SigLIP,0.412772,DINO,0.924615
191,3949,8550,CLIP,0.966498,SigLIP,0.492284,DINO,0.937899
192,6005,8663,CLIP,0.952954,SigLIP,0.432415,DINO,0.919711


In [ ]:
def pipeline(dataset, model_base, processor_base, model_ref_1, processor_ref_1, model_ref_2, processor_ref_2, device, file_name, name_base, name_ref_1, name_ref_2):
  iterator = iter(dataset)
  num = dataset.info.splits['train'].num_examples

  base_embeddings, base_labels, base_faiss_index = build_embeddings(
    dataset_iterator=iterator,
    model=model_base,
    processor=processor_base,
    device=device,
    total_samples=num,
    batch_size=64
  )

  iterator = iter(dataset)

  ref_1_embeddings, ref_1_labels, ref_1_faiss_index = build_embeddings(
    dataset_iterator=iterator,
    model=model_ref_1,
    processor=processor_ref_1,
    device=device,
    total_samples=num,
    batch_size=64
)
  iterator = iter(dataset)

  ref_2_embeddings, ref_2_labels, ref_2_faiss_index = build_embeddings(
    dataset_iterator=iterator,
    model=model_ref_2,
    processor=processor_ref_2,
    device=device,
    total_samples=num,
    batch_size=64
)

  base_k_best_similar = search_similar(base_faiss_index, base_embeddings)
  base_blind_spots = find_blind_spots(base_k_best_similar, ref_1_embeddings, ref_2_embeddings, name_base, name_ref_1, name_ref_2)
  base_blind_spots.to_csv(file_name, sep=';', encoding='utf-8')
  save_path = "/content/drive/MyDrive/vision_blindspots_results"
  full_path = os.path.join(save_path, file_name)
  base_blind_spots.to_csv(full_path, sep=';', encoding='utf-8')


# BASE: CLIP, REF_1: SIGLIP, REF_2: DINOV2, DATASET: CALTECH101

In [ ]:
pipeline(dataset_caltech, clip_model, clip_processor, siglip_model, siglip_processor, dino_model, dino_processor, device, "clip_caltech_blind_spots.csv", 'CLIP', 'SigLIP', 'DINO')

  0%|          | 0/8677 [00:00<?, ?it/s]


Tworzę bazę FAISS
/nLiczba wszystkich przetworzonych zdjęć: 8677


  0%|          | 0/8677 [00:00<?, ?it/s]


Tworzę bazę FAISS
/nLiczba wszystkich przetworzonych zdjęć: 8677


  0%|          | 0/8677 [00:00<?, ?it/s]

AttributeError: 'NoneType' object has no attribute 'ntotal'